In [1]:
import os
import gc
import torch
import numpy as np
import pandas as pd
import shutil
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support
from torch.utils.data import Dataset

# 1. CẤU HÌNH CONFIG CHO XLM-ROBERTA
model_nickname = "XLM-RoBERTa"
model_core_name = "xlm-roberta-base"

kb_paths = {
    "kb1": "/kaggle/input/datasets/kghangco/vn-fb-news-dataset/01_Raw_Data/01_Raw_Data",
    "kb2": "/kaggle/input/datasets/kghangco/vn-fb-news-dataset/02_Basic_Clean/02_Basic_Clean",
    "kb3": "/kaggle/input/datasets/kghangco/vn-fb-news-dataset/03_Full_Clean/03_Full_Clean",
    "kb4": "/kaggle/input/datasets/kghangco/vn-fb-news-dataset/04_No_Stopwords/04_No_Stopwords",
    "kb5": "/kaggle/input/datasets/kghangco/vn-fb-news-dataset/05_Balanced/05_Balanced"
}

all_unique_labels = sorted(["T01", "T02", "T03", "T04", "T05", "T06", "T07", "T08", "T09", "T10", "T11", "T12", "T13", "T14", "T15", "T16", "T17"])
label2id = {label: i for i, label in enumerate(all_unique_labels)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(all_unique_labels)

xlmr_results = []

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"Accuracy": acc, "Precision": precision, "Recall": recall, "F1-macro": f1}

print(f"⏳ Khởi động Tokenizer cho: {model_nickname}...")
tokenizer = AutoTokenizer.from_pretrained(model_core_name)

for kb_name, folder_path in kb_paths.items():
    print("\n" + "="*80)
    print(f"🚀 HUẤN LUYỆN [{model_nickname}] - KỊCH BẢN: [{kb_name.upper()}]")
    print("="*80)
    
    train_path = os.path.join(folder_path, f"{kb_name}_train.csv")
    val_path = os.path.join(folder_path, f"{kb_name}_val.csv")
    test_path = os.path.join(folder_path, f"{kb_name}_test.csv")
    
    if not os.path.exists(train_path):
        print(f"❌ Không tìm thấy file hệ thống cho {kb_name.upper()}. Bỏ qua.")
        continue
        
    df_train = pd.read_csv(train_path)
    df_val = pd.read_csv(val_path)
    df_test = pd.read_csv(test_path)
    
    # 2. KIỂM TRA VÀ CHỌN CỘT VĂN BẢN LOGIC CỦA PHÚC
    available_cols = df_train.columns.tolist()
    if kb_name == 'kb4' and 'text_nosw' in available_cols:
        text_col = 'text_nosw'
    elif kb_name in ['kb2', 'kb3', 'kb5'] and 'text_clean' in available_cols:
        text_col = 'text_clean'
    elif 'post_content_for_labeling' in available_cols:
        text_col = 'post_content_for_labeling'
    else:
        text_col = available_cols[19] 
        
    print(f"🎯 Quyết định: Cột văn bản được chọn là [{text_col}]")
    
    y_train = df_train['topic_label_id'].map(label2id).astype(int).tolist()
    y_val = df_val['topic_label_id'].map(label2id).astype(int).tolist()
    y_test = df_test['topic_label_id'].map(label2id).astype(int).tolist()
    
    train_encodings = tokenizer(df_train[text_col].astype(str).tolist(), truncation=True, padding=True, max_length=256)
    val_encodings = tokenizer(df_val[text_col].astype(str).tolist(), truncation=True, padding=True, max_length=256)
    test_encodings = tokenizer(df_test[text_col].astype(str).tolist(), truncation=True, padding=True, max_length=256)
    
    train_dataset = TextDataset(train_encodings, y_train)
    val_dataset = TextDataset(val_encodings, y_val)
    test_dataset = TextDataset(test_encodings, y_test)
    
    model = AutoModelForSequenceClassification.from_pretrained(model_core_name, num_labels=num_labels)
    
    tmp_output_dir = f"/kaggle/working/tmp_xlmr_{kb_name}"
    
    training_args = TrainingArguments(
        output_dir=tmp_output_dir,
        num_train_epochs=10,              
        per_device_train_batch_size=16,   
        per_device_eval_batch_size=16,
        learning_rate=2e-5,               
        weight_decay=0.01,
        lr_scheduler_type="linear",       
        warmup_ratio=0.1,                                                                                
        eval_strategy="epoch",           
        save_strategy="epoch",            
        save_total_limit=1,                
        load_best_model_at_end=True,   
        metric_for_best_model="F1-macro",
        greater_is_better=True,
        report_to="none"
    )
    
    trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=val_dataset, compute_metrics=compute_metrics)
    trainer.train()
    
    # 💾 3. ĐÓNG GÓI VÀ XUẤT FILE WEIGHT BEST SCENARIO
    print(f"💾 Đang đóng gói và xuất file Weight Best Scenario cho xlmr_{kb_name.upper()}...")
    final_model_save_path = f"/kaggle/working/xlmr_best_model_{kb_name}"
    
    trainer.save_model(final_model_save_path)
    tokenizer.save_pretrained(final_model_save_path)
    
    shutil.make_archive(f"/kaggle/working/xlmr_best_model_{kb_name}", 'zip', final_model_save_path)
    
    # Dọn dẹp folder thô ngay để tránh tràn Disk Quota Kaggle
    if os.path.exists(tmp_output_dir):
        shutil.rmtree(tmp_output_dir)
    if os.path.exists(final_model_save_path):
        shutil.rmtree(final_model_save_path)
    print(f"📦 Đã đóng gói thành công file: xlmr_best_model_{kb_name}.zip")
    
    # 📝 4. ĐÁNH GIÁ VÀ XUẤT REPORT CHI TIẾT TỪNG KỊCH BẢN
    print(f"\n📊 Đang đánh giá trên tập Test của kịch bản {kb_name.upper()}...")
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=1)
    
    target_names = [str(id2label[i]) for i in range(num_labels)]
    cls_report = classification_report(y_test, y_pred, target_names=target_names, digits=4)
    print(cls_report)
    
    # 🌟 THÊM LOGIC: Xuất file report chi tiết dạng .txt cho từng kịch bản
    with open(f"/kaggle/working/xlmr_{kb_name}_report.txt", "w", encoding="utf-8") as f:
        f.write(f"KẾT QUẢ ĐÁNH GIÁ XLM-ROBERTA CHO KỊCH BẢN: {kb_name.upper()}\n")
        f.write("="*60 + "\n")
        f.write(cls_report)
    print(f"📝 Đã xuất file text báo cáo chi tiết từng nhãn: xlmr_{kb_name}_report.txt")
    
    # 🌟 THÊM LOGIC: Tạo và xuất file predictions.csv cho phân tích lỗi giống PhoBERT
    df_test_result = df_test.copy()
    df_test_result['y_true'] = [id2label[i] for i in y_test]
    df_test_result['y_pred'] = [id2label[i] for i in y_pred]
    df_test_result.to_csv(f"/kaggle/working/xlmr_{kb_name}_predictions.csv", index=False)
    print(f"💾 Đã xuất file dự đoán chi tiết tập Test: xlmr_{kb_name}_predictions.csv")
    
    test_metrics = compute_metrics((predictions.predictions, y_test))
    
    xlmr_results.append({
        "Mô hình": model_nickname, 
        "Kịch bản dữ liệu": "KB5 (Oversampling)" if kb_name == "kb5" else kb_name.upper(),
        "Accuracy": round(test_metrics["Accuracy"], 4), 
        "Precision": round(test_metrics["Precision"], 4),
        "Recall": round(test_metrics["Recall"], 4), 
        "F1-macro": round(test_metrics["F1-macro"], 4)
    })
    
    # Giải phóng GPU triệt để sau mỗi kịch bản
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

# ======================================================================
# OUTPUT TỔNG HỢP CUỐI CÙNG CHO XLM-R
# ======================================================================
df_xlmr = pd.DataFrame(xlmr_results)
df_xlmr.to_csv("/kaggle/working/xlmr_report.csv", index=False)

print("\n" + "="*75)
print(f"🎉 HOÀN THÀNH {model_nickname.upper()} TOÀN DIỆN!")
print("Đã lưu file báo cáo tổng hợp xlmr_report.csv, các file predictions.csv, report.txt và 5 file weights zip.")
print("="*75)
print(df_xlmr.to_string(index=False))

⏳ Khởi động Tokenizer cho: XLM-RoBERTa...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

💾 Đang đóng gói và xuất file Weight Best Scenario cho xlmr_KB5...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

📦 Đã đóng gói thành công file: xlmr_best_model_kb5.zip

📊 Đang đánh giá trên tập Test của kịch bản KB5...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

         T01     0.8750    0.8448    0.8596        58
         T02     0.7391    0.7727    0.7556        66
         T03     0.8421    0.8312    0.8366       154
         T04     0.7941    0.8182    0.8060        33
         T05     0.7500    0.8333    0.7895        18
         T06     0.6111    0.6471    0.6286        17
         T07     0.4000    0.5714    0.4706         7
         T08     0.8571    0.8571    0.8571        14
         T09     0.8286    0.7632    0.7945        38
         T10     0.8436    0.8580    0.8507       176
         T11     0.9615    0.9259    0.9434        27
         T12     0.4400    0.5238    0.4783        21
         T13     0.7609    0.6481    0.7000       108
         T14     0.6667    0.6667    0.6667         9
         T15     0.5882    0.5882    0.5882        17
         T16     0.9111    0.9111    0.9111        45
         T17     0.6486    0.8000    0.7164        30

    accuracy              